In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig
import json
import wandb

wandb.init(project='mini-llm', name='DPO')

model_sft_name = "./checkpoints/sft/sft-seed42/final"

with open("dpo_pairs.json") as f:
    data = json.load(f)
print(f"Loaded {len(data['pairs'])} pairs (data cost: {data['metadata']['gpu_hours']:.2f} GPU-hours)")

model = AutoModelForCausalLM.from_pretrained(model_sft_name, dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(model_sft_name)
tokenizer.padding_side = "left"
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
model.resize_token_embeddings(len(tokenizer))

/home/artmak/Desktop/Education/mini-LLM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/artmak/.netrc.
wandb: Currently logged in as: artmak (artmak-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded 2644 pairs (data cost: 1.03 GPU-hours)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 26407.33it/s]


Embedding(151666, 896, padding_idx=151665)

In [2]:
random_state = 42

def format_for_dpo(pair):
    prompt_msgs = [{"role": "user", "content": pair["prompt"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True,
    )
    return {"prompt": prompt_text, "chosen": pair["chosen"], "rejected": pair["rejected"]}

rows = [format_for_dpo(p) for p in data["pairs"]]
dataset = Dataset.from_list(rows).shuffle(seed=random_state)

eval_size = 200
val_dataset = dataset.select(range(eval_size))
train_dataset = dataset.select(range(eval_size, len(dataset)))

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 2444, Val: 200


In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

cfg = DPOConfig(
    output_dir=f"./checkpoints/dpo/dpo-seed{random_state}",
    beta=0.1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    precompute_ref_log_probs=True,
    max_length=1024,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    seed=random_state,
    report_to="wandb",
    run_name=f"dpo-seed{random_state}",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=cfg,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()

final_dir = f"./checkpoints/dpo/dpo-seed{random_state}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model saved to {final_dir}")

Tokenizing train dataset: 100%|██████████| 2444/2444 [00:01<00:00, 1322.96 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 2444/2444 [00:00<00:00, 123598.38 examples/s]
Tokenizing eval dataset: 100%|██████████| 200/200 [00:00<00:00, 1202.02 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 200/200 [00:00<00:00, 84973.74 examples/s]
Flattening the indices: 100%|██████████| 200/200 [00:00<00:00, 242936.81 examples/s]


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
100,0.530227,0.564329,1.282927,520344.000000,-1.478108,-1.575514,0.716842,0.630603,0.326143,0.810000,0.304460,-132.160842,-125.623727
200,0.510223,0.512851,1.229030,1040559.000000,-1.617147,-1.691169,0.716381,0.805432,0.350942,0.855000,0.454490,-130.412555,-125.375740
300,0.481969,0.505040,1.216730,1555580.000000,-1.650119,-1.721076,0.716055,0.819367,0.339862,0.850000,0.479505,-130.273205,-125.486535
306,0.481969,0.503765,1.216876,1583770.000000,-1.649603,-1.720761,0.716285,0.815080,0.331072,0.860000,0.484008,-130.316075,-125.574435


Model saved to ./checkpoints/dpo/dpo-seed42/final


In [ ]:
import os
from peft import PeftModel

adapter_dir = f"./checkpoints/dpo/dpo-seed{random_state}/final"
base_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/sft/sft-seed42/final", dtype=torch.bfloat16
)
base_model.resize_token_embeddings(len(tokenizer))
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()
merged_model = merged_model.to("cuda")

with open("eval_prompts.json") as f:
    eval_prompts = json.load(f)

merged_model.eval()
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id, im_end_id]
results = []

for i, messages in enumerate(eval_prompts[:20]):
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(merged_model.device)

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    results.append({"prompt": messages[0]["content"], "response": response})
    print(f"--- Prompt {i} ---")
    print(f"Q: {messages[0]['content'][:100]}")
    print(f"A: {response[:300]}")
    print()

out_dir = f"./checkpoints/dpo/dpo-seed{random_state}"
merged_model.save_pretrained(os.path.join(out_dir, "merged"))
tokenizer.save_pretrained(os.path.join(out_dir, "merged"))

with open(os.path.join(out_dir, "generations.json"), "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved {len(results)} generations")
print(f"Merged model saved to {out_dir}/merged")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 28615.92it/s]


--- Prompt 0 ---
Q: Mountain Jews or Caucasus Jews also known as Juhuro, Juvuro, Juhuri, Juwuri, Juhurim, Kavkazi Jews o
A: According to the article, women in the Mountain Jewish community learned Hebrew, which is the language of instruction at newly founded elementary schools attended by both Mountain Jewish boys and girls. This policy continued until the beginning of World War II, when schools switched to Russian as th

--- Prompt 1 ---
Q: Alice Geraldine Farrar (February 28, 1882 – March 11, 1967) was an American soprano opera singer and
A: C). No.

The paragraph states that Alice Geraldine Farrar was an American soprano opera singer and film actress, but it does not mention her being an American opera singer in her films. The information given is specific to her singing career as an opera singer in Hollywood films. So, the answer is C

--- Prompt 2 ---
Q: What chemical compound, also known as H2O, is essential for all known forms of life and covers about
A: Water (H2O) is the most 

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.42it/s]

Saved 20 generations
Merged model saved to ./checkpoints/dpo/dpo-seed42/merged


: 